# 🚀 01. Huấn luyện Multi-Modal Teacher Model (Giai đoạn 1B - CMKD)

> **Dự án:** Nhận diện quả sầu riêng chín hay không bằng Chưng cất tri thức Đa phương thức (CMKD)  
> **Phụ trách:** Leader / Member 1 (Vision Lead & Multi-Modal Framework)  
> **Vai trò của Notebook:** Notebook tinh gọn chỉ dùng để:
1. Kết nối Google Drive & kiểm tra GPU.
2. `git clone` / `git pull` mã nguồn mới nhất từ GitHub.
3. Liên kết dữ liệu (`data/final`) và trọng số (`checkpoints/unimodal/{vision,audio,sensor}`) từ Google Drive, đồng thời tạo thư mục đích `checkpoints/teachermodal`.
4. Kích hoạt thực thi file `train_teacher.py` từ thư mục gốc của dự án.

In [ ]:
# ==============================================================================
# 1. KẾT NỐI GOOGLE DRIVE & KIỂM TRA GPU
# ==============================================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    print("✅ Đã kết nối Google Drive thành công!")
else:
    print("ℹ️ Đang chạy trên môi trường Local.")

!nvidia-smi

In [ ]:
# ==============================================================================
# 2. ĐỒNG BỘ CODE MỚI NHẤT TỪ GITHUB
# ==============================================================================
REPO_URL = "https://github.com/Cocobaut/durian-ripeness-cmkd.git"  # Đổi URL repo nếu cần
BRANCH = "cocobaut"
REPO_DIR = "/content/durian-ripeness-cmkd"

if IN_COLAB:
    %cd /content
    if not os.path.exists(REPO_DIR):
        !git clone $REPO_URL $REPO_DIR
        %cd $REPO_DIR
        !git checkout $BRANCH
    else:
        %cd $REPO_DIR
        !git fetch origin
        !git checkout $BRANCH
        !git pull origin $BRANCH
else:
    # Nếu chạy local từ trong subfolder, tự lùi về thư mục gốc của repo
    cur = os.path.abspath(".")
    while not (os.path.isdir(os.path.join(cur, "src")) and os.path.isdir(os.path.join(cur, "workspaces"))):
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    os.chdir(cur)

print(f"✅ Đang làm việc tại thư mục gốc dự án: {os.getcwd()}")

In [ ]:
# ==============================================================================
# 3. CÀI ĐẶT THƯ VIỆN PHỤ THUỘC SIÊU TỐC VỚI `uv`
# ==============================================================================
!pip install -q uv
!uv pip install --system -r requirements.txt
print("✅ Đã hoàn tất cài đặt toàn bộ thư viện!")

In [ ]:
# ==============================================================================
# 4. THIẾT LẬP ĐƯỜNG DẪN CHECKPOINTS & DATASET TỪ GOOGLE DRIVE
# ==============================================================================
# Folder Checkpoints trên Drive: https://drive.google.com/drive/folders/1K6II5aFG31kYrl7rsJj53u2Np4ZzjzG_?usp=drive_link
# Bên trong gồm: unimodal/{vision, audio, sensor} và teachermodal/
from pathlib import Path

if IN_COLAB:
    # Điều chỉnh đường dẫn này khớp với vị trí bạn lưu trên Google Drive
    DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/durian-ripeness-cmkd")
    DRIVE_CHECKPOINTS = DRIVE_PROJECT_ROOT / "checkpoints"
    DRIVE_DATA = DRIVE_PROJECT_ROOT / "data" / "final"

    # Tạo sẵn folder teachermodal trên Google Drive để lưu kết quả trực tiếp
    (DRIVE_CHECKPOINTS / "teachermodal").mkdir(parents=True, exist_ok=True)

    # Tạo symlink từ repo tới Google Drive
    !rm -rf checkpoints/unimodal checkpoints/teachermodal
    !mkdir -p checkpoints data/final
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'unimodal')}" checkpoints/unimodal
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'teachermodal')}" checkpoints/teachermodal
    for split_name in ["train", "valid", "test"]:
        src_split = DRIVE_DATA / split_name
        if src_split.exists():
            !rm -rf data/final/{split_name}
            !ln -sfn "{str(src_split)}" data/final/{split_name}

Path("checkpoints/teachermodal").mkdir(parents=True, exist_ok=True)
print("✅ Kiểm tra trạng thái đường dẫn:")
print(" - Unimodal dir   :", Path("checkpoints/unimodal").resolve())
print(" - Teachermodal   :", Path("checkpoints/teachermodal").resolve())
print(" - Manifest CSV   :", Path("data/final/final_multimodal_manifest.csv").exists())

In [ ]:
# ==============================================================================
# 5. THỰC THI HUẤN LUYỆN MULTI-MODAL TEACHER (2 PHA: WARM-UP FREEZE + JOINT FT)
# ==============================================================================
!python workspaces/member1_vision_lead/04_train_teacher_and_student/train_teacher.py \
    --manifest data/final/final_multimodal_manifest.csv \
    --data-dir data/final \
    --vision-ckpt checkpoints/unimodal/vision/best_vision_unimodal.pth \
    --audio-ckpt checkpoints/unimodal/audio/audio_backbone_teacher.pth \
    --sensor-ckpt checkpoints/unimodal/sensor/best_sensor_unimodal.pth \
    --save-path checkpoints/teachermodal/best_teacher.pth \
    --batch-size 8 \
    --epochs 30 \
    --warmup-epochs 5 \
    --eval-test \
    --device cuda

In [ ]:
# ==============================================================================
# 6. ĐÁNH GIÁ ĐỘC LẬP & XUẤT BÁO CÁO BENCHMARK VỚI EVAL_TEACHER
# ==============================================================================
!python -m src.evaluators.eval_teacher \
    --checkpoint checkpoints/teachermodal/best_teacher.pth \
    --manifest data/final/final_multimodal_manifest.csv \
    --test-dir data/final/test \
    --input-mode multimodal \
    --output-dir workspaces/member1_vision_lead/04_train_teacher_and_student/results

# Nếu chạy trên Colab, tự động sao lưu toàn bộ bảng biểu & đồ thị kết quả lên Google Drive
if IN_COLAB:
    import shutil
    drive_results = DRIVE_PROJECT_ROOT / "results_teacher"
    shutil.copytree(
        "workspaces/member1_vision_lead/04_train_teacher_and_student/results",
        drive_results,
        dirs_exist_ok=True,
    )
    print(f"✅ Đã sao lưu toàn bộ kết quả sang Google Drive tại: {drive_results}")